In [1]:
import pandas as pd
df = pd.read_csv('sample_04.csv')

In [2]:
!pip install scikit-learn

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# -----------------------------
# 1. Copy & clean dataset
# -----------------------------
model_df = df.copy()

# Remove rows without target
model_df = model_df.dropna(subset=['Prospect_Outcome'])

# -----------------------------
# 2. Split X and y
# -----------------------------
X = model_df.drop(columns=['Prospect_Outcome'])
y = model_df['Prospect_Outcome']

# -----------------------------
# 3. Separate column types
# -----------------------------
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include='object').columns

# -----------------------------
# 4. Handle missing values properly
# -----------------------------
X[num_cols] = X[num_cols].fillna(0)
X[cat_cols] = X[cat_cols].fillna('Missing')

# -----------------------------
# 5. Remove high-cardinality columns (VERY IMPORTANT)
# -----------------------------
high_card_cols = [col for col in cat_cols if X[col].nunique() > 50]

print("Dropping high-cardinality columns:", high_card_cols)

X = X.drop(columns=high_card_cols)

# Update categorical columns after drop
cat_cols = X.select_dtypes(include='object').columns

# -----------------------------
# 6. Label Encoding (instead of one-hot)
# -----------------------------
le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le  # store if needed later

# -----------------------------
# 7. Train-test split
# -----------------------------
train_X, test_X, train_y, test_y = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None,
)

# -----------------------------
# 8. Train model (with control)
# -----------------------------
model = DecisionTreeClassifier(
    random_state=42,
    class_weight='balanced',
    max_depth=10,              # prevents overfitting + reduces memory
    min_samples_split=10
)

model.fit(train_X, train_y)

# -----------------------------
# 9. Predictions
# -----------------------------
predictions = model.predict(test_X)

# -----------------------------
# 10. Evaluation
# -----------------------------
print('Train shape:', train_X.shape)
print('Test shape:', test_X.shape)
print('Accuracy:', accuracy_score(test_y, predictions))
print(classification_report(test_y, predictions))

Dropping high-cardinality columns: []
Train shape: (16544, 13)
Test shape: (4137, 13)
Accuracy: 0.871404399323181
              precision    recall  f1-score   support

     Churned       0.40      0.80      0.54       385
         Won       0.98      0.88      0.93      3752

    accuracy                           0.87      4137
   macro avg       0.69      0.84      0.73      4137
weighted avg       0.92      0.87      0.89      4137

